# ETL — equipos — Mantenimiento preventivo de bomba centrífuga

**Fuente:** snapshot `data/raw/equipos_raw.csv` (extraído de PostgreSQL).

Este notebook procesa únicamente la tabla `equipos`: carga el CSV crudo, aplica la transformación y limpieza y exporta el dataframe limpio a `data/processed/`.

## 1. Configuración

Imports, rutas del proyecto y carga de variables de entorno.

In [19]:
from pathlib import Path
import os
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# --- Rutas del proyecto ---
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_CATALOGO_FALLAS = PROJECT_ROOT / "data" / "raw" / "catalogo_fallas_raw.csv"
RAW_DATA_EQUIPOS = PROJECT_ROOT / "data" / "raw" / "equipos_raw.csv"
RAW_DATA_LECTURAS_SENSORES = PROJECT_ROOT / "data" / "raw" / "lecturas_sensores_raw.csv"
RAW_DATA_ORDENES_MANTENIMIENTO = PROJECT_ROOT / "data" / "raw" / "ordenes_mantenimiento_raw.csv"

PROCESSED_FILE = PROCESSED_DIR / "equipos_clean.parquet"



## 2. Análisis, transformación y limpieza

### Tabla equipos_raw

In [20]:
df_equipos = pd.read_csv(RAW_DATA_EQUIPOS)
# df_a (data frame analisis - data set de analisis) copia del dataset para mantener una copia original intacta (df)
df_equipos_w = df_equipos.copy()

df_equipos_w.head()

,equipo_id,codigo_activo,nombre,tipo_equipo,ubicación,fecha_instalacion
0,1,BOMBA-CNT-003,Bomba Centrífuga Principal,Bomba Centrífuga,Galpón 3 - Línea Hidráulica,2024-01-15


In [21]:
df_equipos_w.shape

(1, 6)

In [22]:
df_equipos_w.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   equipo_id          1 non-null      int64
 1   codigo_activo      1 non-null      str  
 2   nombre             1 non-null      str  
 3   tipo_equipo        1 non-null      str  
 4   ubicación          1 non-null      str  
 5   fecha_instalacion  1 non-null      str  
dtypes: int64(1), str(5)
memory usage: 286.0 bytes


In [23]:
print('nulos: ', df_equipos_w.isna().sum().sum())
print('duplicados: ', df_equipos_w.duplicated().sum())

nulos:  0
duplicados:  0


In [24]:
df_equipos_w.columns

Index(['equipo_id', 'codigo_activo', 'nombre', 'tipo_equipo', 'ubicación',
       'fecha_instalacion'],
      dtype='str')

In [25]:
df_equipos_w = df_equipos_w.rename(columns={
    'ubicación' : 'ubicacion'
})

In [26]:
print(df_equipos_w['equipo_id'].unique())
print(df_equipos_w['codigo_activo'].unique())
print(df_equipos_w['nombre'].unique())
print(df_equipos_w['tipo_equipo'].unique())
print(df_equipos_w['ubicacion'].unique())
print(df_equipos_w['fecha_instalacion'].unique())

[1]
<ArrowStringArray>
['  BOMBA-CNT-003 ']
Length: 1, dtype: str
<ArrowStringArray>
[' Bomba Centrífuga Principal  ']
Length: 1, dtype: str
<ArrowStringArray>
['Bomba Centrífuga']
Length: 1, dtype: str
<ArrowStringArray>
['  Galpón 3 - Línea Hidráulica ']
Length: 1, dtype: str
<ArrowStringArray>
['2024-01-15']
Length: 1, dtype: str


In [27]:
cols_texto = ['codigo_activo', 'nombre', 'tipo_equipo', 'ubicacion']

for col in cols_texto:
    df_equipos_w[col] = df_equipos_w[col].str.strip()


In [28]:
print(df_equipos_w['equipo_id'].unique())
print(df_equipos_w['codigo_activo'].unique())
print(df_equipos_w['nombre'].unique())
print(df_equipos_w['tipo_equipo'].unique())
print(df_equipos_w['ubicacion'].unique())
print(df_equipos_w['fecha_instalacion'].unique())

[1]
<ArrowStringArray>
['BOMBA-CNT-003']
Length: 1, dtype: str
<ArrowStringArray>
['Bomba Centrífuga Principal']
Length: 1, dtype: str
<ArrowStringArray>
['Bomba Centrífuga']
Length: 1, dtype: str
<ArrowStringArray>
['Galpón 3 - Línea Hidráulica']
Length: 1, dtype: str
<ArrowStringArray>
['2024-01-15']
Length: 1, dtype: str


In [29]:
df_equipos_w['nombre'] = df_equipos_w['nombre'].replace({
    'Bomba Centrífuga Principal' : 'bomba_centrífuga_principal'
})

df_equipos_w['tipo_equipo'] = df_equipos_w['tipo_equipo'].replace({
    'Bomba Centrífuga' : 'bomba_centrifuga'
})
df_equipos_w['ubicacion'] = df_equipos_w['ubicacion'].replace({
    'Galpón 3 - Línea Hidráulica' : 'galpon_3_Linea_hidraulica'
})

print(df_equipos_w['equipo_id'].unique())
print(df_equipos_w['codigo_activo'].unique())
print(df_equipos_w['nombre'].unique())
print(df_equipos_w['tipo_equipo'].unique())
print(df_equipos_w['ubicacion'].unique())
print(df_equipos_w['fecha_instalacion'].unique())

[1]
<ArrowStringArray>
['BOMBA-CNT-003']
Length: 1, dtype: str
<ArrowStringArray>
['bomba_centrífuga_principal']
Length: 1, dtype: str
<ArrowStringArray>
['bomba_centrifuga']
Length: 1, dtype: str
<ArrowStringArray>
['galpon_3_Linea_hidraulica']
Length: 1, dtype: str
<ArrowStringArray>
['2024-01-15']
Length: 1, dtype: str


In [30]:
df_equipos_w.head()

,equipo_id,codigo_activo,nombre,tipo_equipo,ubicacion,fecha_instalacion
0,1,BOMBA-CNT-003,bomba_centrífuga_principal,bomba_centrifuga,galpon_3_Linea_hidraulica,2024-01-15


In [31]:
df_equipos_w.dtypes

equipo_id            int64
codigo_activo          str
nombre                 str
tipo_equipo            str
ubicacion              str
fecha_instalacion      str
dtype: object

In [32]:
df_equipos_w['fecha_instalacion'] = pd.to_datetime(df_equipos_w['fecha_instalacion'], format='%Y-%m-%d', errors='coerce')


In [33]:
df_equipos_w.dtypes

equipo_id                     int64
codigo_activo                   str
nombre                          str
tipo_equipo                     str
ubicacion                       str
fecha_instalacion    datetime64[us]
dtype: object

In [34]:
print('Resumen del Dataset Work luego de transformaciones y limpieza')
print('=' * 61)
print('DF_Analysis: ', df_equipos.shape, "-> DF_Work: ", df_equipos_w.shape)
print('=' * 61)
print('Columnas antes: ', list(df_equipos.columns))
print('Columnas después:', list(df_equipos_w.columns))
print('=' * 61)
print('DF_Work tiene: ', df_equipos_w.isna().sum().sum(), ' Valores Nulos')
print('DF_Work tiene: ', df_equipos_w.duplicated().sum(), ' Valores Duplicados')
print('=' * 61)
print('Tipos de datos finales:')
print(df_equipos_w.dtypes.to_string())
print('=' * 61)

Resumen del Dataset Work luego de transformaciones y limpieza
DF_Analysis:  (1, 6) -> DF_Work:  (1, 6)
Columnas antes:  ['equipo_id', 'codigo_activo', 'nombre', 'tipo_equipo', 'ubicación', 'fecha_instalacion']
Columnas después: ['equipo_id', 'codigo_activo', 'nombre', 'tipo_equipo', 'ubicacion', 'fecha_instalacion']
DF_Work tiene:  0  Valores Nulos
DF_Work tiene:  0  Valores Duplicados
Tipos de datos finales:
equipo_id                     int64
codigo_activo                   str
nombre                          str
tipo_equipo                     str
ubicacion                       str
fecha_instalacion    datetime64[us]


In [35]:
df_equipos_w.head()

,equipo_id,codigo_activo,nombre,tipo_equipo,ubicacion,fecha_instalacion
0,1,BOMBA-CNT-003,bomba_centrífuga_principal,bomba_centrifuga,galpon_3_Linea_hidraulica,2024-01-15


## 3. Exportación a Parquet

Se guarda el dataframe limpio `df_equipos_w` en `data/processed/equipos_clean.parquet`.

In [36]:
# Exportación del dataframe limpio a Parquet
archivo_temporal = PROCESSED_FILE.with_suffix(".parquet.tmp")

df_equipos_w.to_parquet(archivo_temporal, index=False)
archivo_temporal.replace(PROCESSED_FILE)

print(f"Filas: {len(df_equipos_w):,} | Columnas: {df_equipos_w.shape[1]}")

Filas: 1 | Columnas: 6
